In [2]:
print("hello dhiraj")

hello dhiraj


In [3]:
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] += r";C:\hadoop\bin"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("OrderStreamingNotebook") \
    .master("local[*]") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
    ) \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("Spark session started")

Spark session started


In [5]:
order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("event_time", StringType(), True)
])

print("Schema created")

Schema created


In [6]:
order_schema

StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('product_id', StringType(), True), StructField('event_type', StringType(), True), StructField('quantity', IntegerType(), True), StructField('price', DoubleType(), True), StructField('event_time', StringType(), True)])

In [7]:
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "127.0.0.1:9092") \
    .option("subscribe", "order_events") \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

print("Kafka stream connected")

Kafka stream connected


In [8]:

raw_df = df.selectExpr(
    "CAST(key AS STRING) as kafka_key",
    "CAST(value AS STRING) as json_data",
    "topic",
    "partition",
    "offset",
    "timestamp"
)

print("Raw dataframe created")

Raw dataframe created


In [11]:

raw_query = raw_df.writeStream \
    .format("memory") \
    .queryName("raw_orders") \
    .outputMode("append") \
    .start()

print("Memory stream started")

Memory stream started


In [7]:

raw_query.awaitTermination(5)

False

In [8]:

spark.sql("SELECT * FROM raw_orders LIMIT 10").show(truncate=False)

+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------+------+-----------------------+
|kafka_key|json_data                                                                                                                                                                        |topic       |partition|offset|timestamp              |
+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------+------+-----------------------+
|NULL     |{"order_id": "order_1", "customer_id": "cust_19", "product_id": "prod_43", "event_type": "CREATED", "quantity": 1, "price": 49.52, "event_time": "2026-03-31T05:24:05.548161"}   |order_events|0        |79717 |2026-03-31 10:55:19.548|
|NULL     |{"order_id": 